<a href="https://colab.research.google.com/github/yoshiagent/vibration-colab/blob/main/notebooks/Harmonic_Excitation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 簡諧激振 Harmonic Excitation

對照網頁：[振動學｜簡諧激振](https://yoshiagent.github.io/vibration/HX.html)

| 原檔 | 系統 | 本筆記本 |
|---|---|---|
| `freqrespord2.m` | 二階系統 Bode 圖＋弦波輸入的 `lsim` | 第 1 節 |
| `mck.m` | 質量阻尼彈簧受力 $f_0\sin\omega t$ 的穩態響應 | 第 2 節 |
| `seis.m` | 基座激振 $y=Y\sin\omega t$ | 第 3 節 |
| `transm.m` | 力傳遞率（傳到地面的力） | 第 4 節 |
| `rotorunb.m` | 旋轉不平衡 | 第 5 節 |

原程式的做法都一樣：寫出轉移函數 → `bode` 畫頻率響應 → `freqresp` 取 $G(j\omega)$ → 穩態響應＝輸入振幅 × $|G(j\omega)|$、相位加上 $\angle G(j\omega)$。
這裡用 `scipy.signal` 對應（`freqresp` → `signal.freqresp`），並用 `solve_ivp` 長時間積分確認暫態消失後確實是這個穩態解。

In [1]:
import numpy as np
from scipy import signal
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def bode(num, den, w_lo, w_hi, title, mark=None):
    """對應 MATLAB bode(sys,{w_lo,w_hi})：上圖增益（dB）、下圖相位（deg），mark 為激振頻率"""
    w = np.logspace(np.log10(w_lo), np.log10(w_hi), 600)
    _, G = signal.freqresp(signal.TransferFunction(num, den), w)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06)
    fig.add_trace(go.Scatter(x=w, y=20 * np.log10(np.abs(G)), name="|G| (dB)"), 1, 1)
    fig.add_trace(go.Scatter(x=w, y=np.degrees(np.unwrap(np.angle(G))), name="phase (deg)"), 2, 1)
    if mark is not None:
        _, Gm = signal.freqresp(signal.TransferFunction(num, den), [mark])
        fig.add_trace(go.Scatter(x=[mark], y=[20 * np.log10(abs(Gm[0]))], mode="markers", marker=dict(size=10, color="red"), name="w"), 1, 1)
        fig.add_trace(go.Scatter(x=[mark], y=[np.degrees(np.angle(Gm[0]))], mode="markers", marker=dict(size=10, color="red"), showlegend=False), 2, 1)
    fig.update_xaxes(type="log", title_text="w (rad/s)", row=2, col=1)
    fig.update_yaxes(title_text="Magnitude (dB)", row=1, col=1)
    fig.update_yaxes(title_text="Phase (deg)", row=2, col=1)
    fig.update_layout(title=title, height=520, template="plotly_white", margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


def freqresp(num, den, w):
    """對應 MATLAB freqresp(sys,w)：回傳 G(jw)"""
    return signal.freqresp(signal.TransferFunction(num, den), [w])[1][0]


def steady_check(label, num, den, u, w, amp, phase, kind="sin", T=None):
    """把 u(t) 餵給轉移函數做長時間模擬，比對最後兩個週期與 amp*sin(wt+phase)"""
    period = 2 * np.pi / w
    T = T or 60 * period
    t = np.linspace(0, T, max(40001, int(T / period * 800) + 1))       # 每個週期至少 800 點（lsim 以線性內插輸入）
    _, y, _ = signal.lsim(signal.TransferFunction(num, den), u(t), t)
    tail = t > T - 2 * period
    ys = amp * (np.sin if kind == "sin" else np.cos)(w * t + phase)
    rel = np.max(np.abs(y[tail] - ys[tail])) / amp
    print(f"{label}: steady-state amplitude = {amp:.6g}, phase = {np.degrees(phase):.2f} deg; lsim tail vs formula: rel. error = {rel:.1e}")
    return t, y


def plot_pair(t, a, b, la, lb, title, ylab):
    fig = go.Figure([go.Scatter(x=t, y=a, name=la, line=dict(color="blue")), go.Scatter(x=t, y=b, name=lb, line=dict(color="red"))])
    fig.update_layout(title=title, xaxis_title="t (s)", yaxis_title=ylab, height=400, template="plotly_white", margin=dict(l=60, r=20, t=50, b=50))
    fig.show()

## 1. 二階系統的頻率響應（`freqrespord2.m`）

```matlab
wn=1; z=0.3;
[num den]=ord2(wn,z);     % num = 1, den = [1 2*z*wn wn^2]
sys=tf(num,den);  bode(sys)
t=0:0.01:30; w=2; u=sin(w*t);  lsim(sys,u,t)
```
注意 `ord2` 的分子是 1（不是 $\omega_n^2$），這裡 $\omega_n=1$ 所以兩者相同。

In [2]:
wn, z, w = 1.0, 0.3, 2.0
num, den = [1.0], [1.0, 2 * z * wn, wn**2]
bode(num, den, 0.1, 10, "Bode plot, wn = 1, zeta = 0.3 (freqrespord2.m)", mark=w)
G = freqresp(num, den, w)
r = w / wn
print(f"|G(j2)| = {abs(G):.4f}  (formula 1/sqrt((1-r^2)^2+(2 z r)^2) = {1 / np.sqrt((1 - r**2)**2 + (2 * z * r)**2):.4f}), angle = {np.degrees(np.angle(G)):.2f} deg")
t = np.arange(0, 30 + 1e-9, 0.01)
_, y, _ = signal.lsim(signal.TransferFunction(num, den), np.sin(w * t), t)
plot_pair(t, np.sin(w * t), y, "u = sin 2t", "y (lsim)", "lsim(sys, sin 2t) — transient, then steady state", "y")
steady_check("freqrespord2", num, den, lambda t: np.sin(w * t), w, abs(G), np.angle(G))
print("note: w > wn, so the output lags by more than 90 deg (np.angle / atan2 gives the correct quadrant).")

|G(j2)| = 0.3095  (formula 1/sqrt((1-r^2)^2+(2 z r)^2) = 0.3095), angle = -158.20 deg


freqrespord2: steady-state amplitude = 0.309492, phase = -158.20 deg; lsim tail vs formula: rel. error = 5.1e-06
note: w > wn, so the output lags by more than 90 deg (np.angle / atan2 gives the correct quadrant).


## 2. 質量阻尼彈簧受弦波力（`mck.m`）

```matlab
m=10; c=8; k=1;
wn=sqrt(k/m), z=c/m/2/wn
w=10; f0=5;
sys=tf(1,[m c k]);  bode(sys,{w/10,w*10})
sysw=freqresp(sys,w);  syswabs=abs(sysw)
ymass=f0*syswabs*sin(w*t+syswphase);
plot(t,f,'b',t,ymass,'r')
```

In [3]:
m, c, k, w, f0 = 10.0, 8.0, 1.0, 10.0, 5.0
wn = np.sqrt(k / m); z = c / m / 2 / wn
print(f"wn = {wn:.4f} rad/s, z = {z:.4f}  (z > 1: overdamped)")
num, den = [1.0], [m, c, k]
bode(num, den, w / 10, w * 10, "Bode plot of 1/(m s^2 + c s + k) (mck.m)", mark=w)
G = freqresp(num, den, w)
print(f"syswabs = |G(j10)| = {abs(G):.6f}  (formula 1/sqrt((k-m w^2)^2+(c w)^2) = {1 / np.hypot(k - m * w**2, c * w):.6f})")
period = 2 * np.pi / w
t = np.arange(0, 2 * period + 1e-12, 0.01 * period)
plot_pair(t, f0 * np.sin(w * t), f0 * abs(G) * np.sin(w * t + np.angle(G)), "f = 5 sin 10t", "ymass (steady state)", "Force and mass displacement (mck.m)", "")
steady_check("mck", num, den, lambda t: f0 * np.sin(w * t), w, f0 * abs(G), np.angle(G), T=150)   # 過阻尼的慢極點 -0.155，暫態要較久才消失
print("w = 10 >> wn = 0.316: the mass is mass-controlled, x ~ f0/(m w^2) = %.6f and almost 180 deg out of phase." % (f0 / (m * w**2)))

wn = 0.3162 rad/s, z = 1.2649  (z > 1: overdamped)


syswabs = |G(j10)| = 0.000998  (formula 1/sqrt((k-m w^2)^2+(c w)^2) = 0.000998)


mck: steady-state amplitude = 0.00498903, phase = -175.42 deg; lsim tail vs formula: rel. error = 5.1e-06
w = 10 >> wn = 0.316: the mass is mass-controlled, x ~ f0/(m w^2) = 0.005000 and almost 180 deg out of phase.


## 3. 基座激振（`seis.m`）

```matlab
m=2; c=80; k=20000;  wn=sqrt(k/m); z=c/m/2/wn;
w=80; ybasemag=5;
sys=tf([2*z*wn wn^2],[1 2*z*wn wn^2]);      % X(s)/Y(s)
ymass=ybasemag*syswabs*sin(w*t+syswphase);
```
網頁講義：$\dfrac{x_0}{y_0}=\dfrac{\sqrt{1+(2\zeta r)^2}}{\sqrt{(1-r^2)^2+(2\zeta r)^2}}$，$r=\omega/\omega_n$。

In [4]:
m, c, k, w, Y = 2.0, 80.0, 20000.0, 80.0, 5.0
wn = np.sqrt(k / m); z = c / m / 2 / wn; r = w / wn
num, den = [2 * z * wn, wn**2], [1.0, 2 * z * wn, wn**2]
print(f"wn = {wn:.1f} rad/s, z = {z:.2f}, r = w/wn = {r:.2f}")
bode(num, den, w / 10, w * 10, "Seismic excitation X/Y (seis.m)", mark=w)
G = freqresp(num, den, w)
formula = np.sqrt(1 + (2 * z * r)**2) / np.sqrt((1 - r**2)**2 + (2 * z * r)**2)
print(f"syswabs = {abs(G):.4f}  (formula = {formula:.4f}), mass amplitude = {Y * abs(G):.4f} (base amplitude 5)")
phase = np.arctan(2 * z * r) - np.arctan2(2 * z * r, 1 - r**2)
print(f"phase = {np.degrees(np.angle(G)):.2f} deg  (formula atan(2zr) - atan2(2zr, 1-r^2) = {np.degrees(phase):.2f} deg)")
period = 2 * np.pi / w
t = np.linspace(0, 2 * period, 101)
plot_pair(t, Y * np.sin(w * t), Y * abs(G) * np.sin(w * t + np.angle(G)), "ybase", "ymass", "Base and mass motion (seis.m)", "displacement")
steady_check("seis", num, den, lambda t: Y * np.sin(w * t), w, Y * abs(G), np.angle(G))

wn = 100.0 rad/s, z = 0.20, r = w/wn = 0.80


syswabs = 2.1798  (formula = 2.1798), mass amplitude = 10.8992 (base amplitude 5)
phase = -23.89 deg  (formula atan(2zr) - atan2(2zr, 1-r^2) = -23.89 deg)


seis: steady-state amplitude = 10.8992, phase = -23.89 deg; lsim tail vs formula: rel. error = 5.1e-06


(array([0.00000000e+00, 9.81747704e-05, 1.96349541e-04, ...,
        4.71219263e+00, 4.71229081e+00, 4.71238898e+00], shape=(48001,)),
 array([ 0.00000000e+00,  7.76342375e-05,  3.12637989e-04, ...,
        -4.56975664e+00, -4.49190207e+00, -4.41377041e+00], shape=(48001,)))

## 4. 力傳遞率（`transm.m`）

```matlab
m=10; c=8; k=1;  w=1; f0=5;
sys=tf([c k],[m c k]);        % F_T(s)/F(s)
ft=f0*syswabs*sin(w*t+syswphase);
sys1=tf(1,[m c k]);           % X(s)/F(s)
ymass=f0*syswabs1*sin(w*t+syswphase1);
```
網頁講義：$TR=\dfrac{f_{T0}}{f_0}=\dfrac{\sqrt{1+(2\zeta r)^2}}{\sqrt{(1-r^2)^2+(2\zeta r)^2}}$，與基座激振的位移比是同一個式子；$r>\sqrt2$ 時 $TR<1$。

In [5]:
m, c, k, w, f0 = 10.0, 8.0, 1.0, 1.0, 5.0
wn = np.sqrt(k / m); z = c / m / 2 / wn; r = w / wn
print(f"wn = {wn:.4f}, z = {z:.4f}, r = {r:.4f}")
bode([c, k], [m, c, k], w / 10, w * 10, "Force transmissibility F_T/F (transm.m)", mark=w)
G = freqresp([c, k], [m, c, k], w)
TR = np.sqrt(1 + (2 * z * r)**2) / np.sqrt((1 - r**2)**2 + (2 * z * r)**2)
print(f"TR = |T(j1)| = {abs(G):.4f}  (formula = {TR:.4f}), transmitted force amplitude = {f0 * abs(G):.4f}")
G1 = freqresp([1.0], [m, c, k], w)
print(f"mass amplitude = f0 |X/F| = {f0 * abs(G1):.4f}")
steady_check("transm F_T", [c, k], [m, c, k], lambda t: f0 * np.sin(w * t), w, f0 * abs(G), np.angle(G), T=200)
rr = np.linspace(0.01, 4, 800)
fig = go.Figure()
for zz in (0.05, 0.1, 0.25, 0.5, 1.0):
    fig.add_trace(go.Scatter(x=rr, y=np.sqrt(1 + (2 * zz * rr)**2) / np.sqrt((1 - rr**2)**2 + (2 * zz * rr)**2), name=f"zeta = {zz}"))
fig.add_vline(x=np.sqrt(2), line_dash="dot", annotation_text="r = sqrt(2)")
fig.update_layout(title="Transmissibility vs frequency ratio", xaxis_title="r = w/wn", yaxis_title="TR", yaxis_range=[0, 5],
                  height=420, template="plotly_white")
fig.show()
print(f"all curves pass TR = 1 at r = sqrt(2): " +
      ", ".join(f"{np.sqrt(1 + (2 * zz * np.sqrt(2))**2) / np.sqrt(1 + (2 * zz * np.sqrt(2))**2):.3f}" for zz in (0.05, 0.1, 0.25, 0.5, 1.0)))

wn = 0.3162, z = 1.2649, r = 3.1623


TR = |T(j1)| = 0.6695  (formula = 0.6695), transmitted force amplitude = 3.3477
mass amplitude = f0 |X/F| = 0.4152
transm F_T: steady-state amplitude = 3.34767, phase = -55.49 deg; lsim tail vs formula: rel. error = 2.1e-06


all curves pass TR = 1 at r = sqrt(2): 1.000, 1.000, 1.000, 1.000, 1.000


## 5. 旋轉不平衡（`rotorunb.m`）

```matlab
m2=2; m1=10; e=1; c=8; k=50; w=1;
f0=2*m2/(m1+2*m2)*e*w*w
wn=sqrt(k/(m1+2*m2)),  z=c/(m1+2*m2)/2/wn
sys=tf([1],[1 2*z*wn wn^2]);
tf=2*wperiod;                 % ← 這裡把 tf 函數名稱當成變數用了（之後就不能再呼叫 tf()）
ymass=f0*syswabs*cos(w*t+syswphase);
```
網頁講義（已加編註更正）：$x_0=\dfrac{2m_2e}{m_1+2m_2}\,\dfrac{(\omega/\omega_n)^2}{\sqrt{(1-r^2)^2+(2\zeta r)^2}}$——分子是 **平方**。

In [6]:
m2, m1, e, c, k, w = 2.0, 10.0, 1.0, 8.0, 50.0, 1.0
M = m1 + 2 * m2
f0 = 2 * m2 / M * e * w * w
wn = np.sqrt(k / M); z = c / M / 2 / wn; r = w / wn
print(f"f0 = {f0:.4f}, wn = {wn:.4f}, z = {z:.4f}, r = {r:.4f}")
num, den = [1.0], [1.0, 2 * z * wn, wn**2]
bode(num, den, w / 10, w * 10, "Rotating unbalance X/F (rotorunb.m)", mark=w)
G = freqresp(num, den, w)
x0 = f0 * abs(G)
x0_fixed = 2 * m2 * e / M * r**2 / np.sqrt((1 - r**2)**2 + (2 * z * r)**2)
x0_orig = 2 * m2 * e / M * r / np.sqrt((1 - r**2)**2 + (2 * z * r)**2)
print(f"x0 = f0 |G(jw)| = {x0:.6f};  corrected formula (r^2) = {x0_fixed:.6f};  original slide (r) = {x0_orig:.6f}")
steady_check("rotorunb", num, den, lambda t: f0 * np.cos(w * t), w, x0, np.angle(G), kind="cos")
rr = np.linspace(0.01, 5, 800)
fig = go.Figure()
for zz in (0.05, 0.15, 0.3, 0.7):
    fig.add_trace(go.Scatter(x=rr, y=rr**2 / np.sqrt((1 - rr**2)**2 + (2 * zz * rr)**2), name=f"zeta = {zz}"))
fig.add_hline(y=1, line_dash="dot", annotation_text="x0 -> 2 m2 e/(m1+2m2)")
fig.update_layout(title="Rotating unbalance: x0 (m1+2m2)/(2 m2 e) vs r", xaxis_title="r = w/wn", yaxis_title="normalized amplitude",
                  yaxis_range=[0, 5], height=420, template="plotly_white")
fig.show()

f0 = 0.2857, wn = 1.8898, z = 0.1512, r = 0.5292


x0 = f0 |G(jw)| = 0.108465;  corrected formula (r^2) = 0.108465;  original slide (r) = 0.204980


rotorunb: steady-state amplitude = 0.108465, phase = -12.53 deg; lsim tail vs formula: rel. error = 5.1e-06
